# Data modeling 8: physical design and performance

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-08-physical-design-and-performance.ipynb)

The logical model says what a table means. The physical design decides how it is stored: which columns it
is partitioned and sorted by, how big its files are, which aggregates are precomputed, and which data types
hold its keys and amounts. Those choices decide how much data each query reads, and in a cloud warehouse what
it costs.

The shop's real tables are too small to show this, so the lesson generates a year of synthetic order lines (one
million rows) and measures what queries read. The measurements come from DuckDB's `explain analyze`, which
reports the files and row groups a scan touched; timings vary between runs and are left out.

**Prerequisites:** [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb) and [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb). Some familiarity with [Parquet](../parquet/parquet.ipynb) helps.

*Last reviewed: November 2026, against DuckDB 2.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_products` (the product catalogue): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
""")
con.sql("show tables").show()

The next cell generates one million order lines spread over 2025. `hash()` makes the values look random but be the
same on every run. It also defines `scan_report`, which runs a query under `explain analyze` and prints only the lines
that say how much data the scan touched.

In [ ]:
import tempfile

WORK_DIR = tempfile.mkdtemp(prefix="modeling-")  # Parquet files written by this lesson

con.sql("""
create table fact_sales as
select i                                                       as line_id,
       date '2025-01-01' + cast(hash(i) % 365 as int)          as order_date,
       cast(hash(i * 7) % 50000 as int)                        as customer_id,
       'SKU-0' || cast(1 + hash(i * 13) % 8 as int)            as sku,
       cast(1 + hash(i * 17) % 5 as int)                       as quantity,
       cast((hash(i * 19) % 5000) / 100.0 as decimal(10, 2))   as net_amount
from range(1000000) t(i)
""")


def scan_report(query):
    """Run the query under EXPLAIN ANALYZE and print the scan's file and row-group counts."""
    plan = con.sql("explain analyze " + query).fetchall()[0][1]
    for line in plan.splitlines():
        text = line.strip("│ ").strip()
        if text.startswith(("Scanning Files", "Row Groups Scanned", "File Filters")):
            print("  " + text)


con.sql("select count(*) as rows, min(order_date) as first_day, max(order_date) as last_day from fact_sales").show()

## How analytical engines read data

Warehouses and lakehouse formats store data **by column**, compressed, in large blocks: Parquet **row groups**
and files, Snowflake **micro-partitions**, BigQuery blocks. Each block records the minimum and maximum of every
column (a **zone map**). A query then reads:

- only the **columns** it names, so a wide table costs nothing extra for columns a query skips;
- only the **blocks** whose min/max range could contain matching rows, so a filter on a column the data is
  organized by skips most of the table.

The two physical design levers follow from that: **partitioning** splits a table into separate files or
directories by a column's value, so whole partitions are skipped; **clustering** (sorting) keeps similar values
together inside and across files, so zone maps can skip blocks. Both only help queries that filter on those
columns, so choose them from the filters your queries actually use, usually a date.

**Where it matters**
- Fact tables of more than a few gigabytes; cost control in pay-per-byte warehouses such as BigQuery.

**Watch out for**
- Choosing a partition or cluster column that queries rarely filter on: all the cost of maintaining it, none of the benefit.

## Partitioning

A partitioned table stores each value of the partition column (usually a day or month of the event date) in
its own directory or partition. A query that filters on that column reads only the matching partitions: this is
**partition pruning**.

Pick a partition column that almost every query filters on, with a moderate number of values: a date at day or
month granularity is the common choice. Too many partitions (by customer, by hour on a small table) produce
thousands of tiny files, and every query pays to list and open them. Each engine has guidance on partition size,
typically aiming for files of hundreds of megabytes to a gigabyte.

**Where it matters**
- Time-ranged facts and event tables, and lifecycle management (dropping or archiving old partitions).

**Watch out for**
- Filtering on an expression of the partition column (`year(order_date) = 2025` instead of a range on `order_date`): some engines cannot prune through the function.
- Partitioning by a high-cardinality column: the small-files problem.

### Example

Write the fact as Parquet partitioned by month, then read one month with and without using the partition column.

In [ ]:
con.sql(f"""
copy (select *, strftime(order_date, '%Y-%m') as order_month from fact_sales)
to '{WORK_DIR}/sales_by_month' (format parquet, partition_by (order_month))
""")
files = con.sql(f"select count(*) from glob('{WORK_DIR}/sales_by_month/*/*.parquet')").fetchall()[0][0]
print(f"{files} files, one directory per month, e.g. sales_by_month/order_month=2025-03/")

source = f"read_parquet('{WORK_DIR}/sales_by_month/*/*.parquet', hive_partitioning = true)"
print("filter on the partition column:")
scan_report(f"select sum(net_amount) from {source} where order_month = '2025-03'")
print("filter on order_date only:")
scan_report(f"select sum(net_amount) from {source} where order_date between '2025-03-01' and '2025-03-31'")

**What to notice:** Filtering on `order_month` turned into a **file filter**: DuckDB opened 1 of the 12 files and never touched the
other eleven. Filtering on `order_date` asks the same question, but the engine does not know that `order_date`
and the directory name are related, so there is no file filter. It had to open all 12 files and read each one's
statistics before skipping the row groups that could not match (`1 / 12` scanned). On a real table with thousands
of files, opening every file is the expensive part. Query by the partition column, or use a format such as Iceberg whose hidden partitioning
derives the partition from `order_date` for you.

## Clustering and zone maps

Inside a table, the order of rows decides how useful zone maps are. If rows are written in random order, every
block holds dates from the whole year, every block's min/max range overlaps any date filter, and nothing can be
skipped. Sorted by date, each block covers a narrow range and a one-week filter touches one or two blocks.

Warehouses expose this as **clustering**: Snowflake clustering keys, BigQuery clustered tables, Redshift sort
keys, Delta Z-ordering and liquid clustering, Iceberg sort orders. Sorting by two or three columns that are
often filtered together (date, then customer) helps both; past that, the later columns gain little.

**Where it matters**
- Large facts filtered by date ranges, customer or tenant; dimensions looked up by key.

**Watch out for**
- Clustering decays as new unsorted data arrives; warehouses recluster in the background, and that costs compute.

### Example

Write the same rows twice, once in arrival order and once sorted by date, in row groups of 100,000 rows. Compare their zone maps and a one-week query.

In [ ]:
con.sql(f"copy fact_sales to '{WORK_DIR}/unsorted.parquet' (format parquet, row_group_size 100000)")
con.sql(f"copy (from fact_sales order by order_date) to '{WORK_DIR}/sorted.parquet' (format parquet, row_group_size 100000)")
con.sql(f"""
select replace(file_name, '{WORK_DIR}/', '') as file, row_group_id, stats_min as min_date, stats_max as max_date
from parquet_metadata('{WORK_DIR}/*sorted.parquet')
where path_in_schema = 'order_date' and row_group_id < 3
order by file desc, row_group_id
""").show()
for name in ["unsorted", "sorted"]:
    print(f"{name}: one week in June")
    scan_report(f"select sum(net_amount) from '{WORK_DIR}/{name}.parquet' where order_date between '2025-06-02' and '2025-06-08'")

**What to notice:** In the unsorted file every row group spans the whole year (January 1 to December 31), so a one-week filter
cannot rule any of them out: all 10 row groups are scanned. In the sorted file each row group covers about five
weeks, and the same filter scans 1. Same data, same query, a tenth of the reading, purely from the write order.

## Aggregate tables and materialized views

Dashboards ask the same coarse questions repeatedly: sales by day and product, by month and region. An
**aggregate table** stores the fact pre-summarized at that coarser grain, so those queries read thousands of
rows instead of millions. A **materialized view** is the same idea managed by the engine (Snowflake, BigQuery,
Redshift, Databricks), which keeps it current and can rewrite matching queries to use it automatically.

Rules that keep aggregates correct:

- Aggregate only **additive** measures, or store the components (sum and count, never an average).
- Keep the aggregate's grain explicit and its dimension keys conformed with the base fact, so it can be joined
  to the same dimensions.
- Refresh it with the same incremental, idempotent pattern as the fact (lesson 7): recompute affected days.
- Reconcile it against the base fact in tests (lesson 9).

**Where it matters**
- High-traffic dashboards, BI extracts, and any metric queried far more often than its data changes.

**Watch out for**
- Distinct counts in aggregates: they cannot be summed across rows; store a sketch (HyperLogLog) or recompute from the base.

### Example

Build a daily aggregate by product, then answer the same monthly question from the base fact and from the aggregate.

In [ ]:
con.sql("""
create table agg_sales_daily as
select order_date, sku, count(*) as lines, sum(quantity) as units, sum(net_amount) as net_sales
from fact_sales group by all
""")
rows = con.sql("select (select count(*) from fact_sales), (select count(*) from agg_sales_daily)").fetchall()[0]
print(f"base fact: {rows[0]:,} rows; daily aggregate: {rows[1]:,} rows")

monthly = """select date_trunc('month', order_date)::date as month, {cols}
             from {table} where order_date < '2025-04-01' group by all order by month"""
con.sql(monthly.format(table="fact_sales", cols="count(*) as lines, sum(net_amount) as net_sales")).show()
con.sql(monthly.format(table="agg_sales_daily", cols="sum(lines) as lines, sum(net_sales) as net_sales")).show()

**What to notice:** The aggregate has 2,920 rows (365 days × 8 SKUs) instead of 1,000,000, and gives identical monthly results,
because counts and sums add up across days. Note `sum(lines)`, not `count(*)`, when reading the aggregate: each of
its rows already stands for many lines.

## Keys and data types

Physical types matter more at scale than they look:

- **Join keys:** integer surrogate keys are smaller and faster to join and compare than long strings or
  composite natural keys. Hash keys (Data Vault) trade some of that for parallel loading.
- **Money:** use `decimal`, never floating point. Floating-point sums accumulate rounding error, and the error
  changes with the order rows are added in, so the same query can return different cents on different runs.
- **Dates and timestamps:** store event time as a timestamp with a known time zone (usually UTC) and derive the
  date the business means in the model, not in every report.
- **Low-cardinality text** (status, country) compresses well in columnar formats; a lookup table is rarely needed
  for storage reasons.

**Where it matters**
- Every fact table's DDL; reviewing models generated by tools that default to strings or floats.

**Watch out for**
- Keys stored as different types in different tables (`int` here, `varchar` there): joins need casts and may stop using statistics.

### Example

Sum one million amounts as `decimal` and as `double`, adding the rows in two different orders.

In [ ]:
con.sql("""
select 'decimal(10,2)' as type,
       cast(sum(net_amount order by line_id) as varchar)       as sum_in_id_order,
       cast(sum(net_amount order by line_id desc) as varchar)  as sum_in_reverse_order
from fact_sales
union all
select 'double',
       cast(sum(net_amount::double order by line_id) as varchar),
       cast(sum(net_amount::double order by line_id desc) as varchar)
from fact_sales
""").show()

**What to notice:** The decimal sums are exact and identical whichever order the rows are added in. The double sums miss the exact
25007501.05 in the last digits, and miss it differently depending on the order, because each floating-point
addition rounds. Here the error is below a cent, but it grows with the number of rows and with arithmetic on the
sums, it makes equality checks between two totals fail, and in a parallel engine the order of additions changes
from run to run, so the same report can differ between refreshes with no change in the data.

## Physical design in each warehouse

The same ideas go by different names:

| Engine | Skip partitions | Organize within | Notes |
|---|---|---|---|
| Snowflake | (automatic micro-partitions) | clustering key | micro-partitions are automatic; define a clustering key only for large tables filtered on a column the load order does not follow |
| BigQuery | `partition by` a date or integer range | `cluster by` up to 4 columns | billing by bytes scanned makes pruning a cost question; `require_partition_filter` enforces it |
| Redshift | (none) | sort key; distribution key | `distkey` co-locates join keys across nodes; `diststyle all` copies small dimensions to every node |
| Databricks / Delta | partition columns (large tables only) | liquid clustering or Z-order | liquid clustering replaces partitioning plus Z-order for most new tables |
| Iceberg | hidden partitioning by transforms (`days(ts)`) | sort order | queries filter on the column, not a derived partition column |
| DuckDB / Parquet files | Hive-style directories | write order, row group size | what this lesson used |

See the [Snowflake](../snowflake/snowflake.ipynb), [BigQuery](../bigquery/bigquery.ipynb),
[Redshift](../redshift/redshift.ipynb), [Delta Lake](../delta/delta.ipynb) and [Iceberg](../iceberg/iceberg.ipynb)
notebooks for each engine's details.

## Stars, wide tables and the engine

Physical design also informs the logical shape. A star needs a join per dimension at query time; a wide table
(lesson 6) does not. Modern columnar engines join a large fact to small dimensions cheaply (the dimensions fit in
memory and are broadcast), so for most workloads a star performs well and keeps the model flexible. Wide tables
win when dimensions are large, when the engine is weak at joins, or when one known query must be as fast as
possible, and they cost more to keep current when dimension attributes change.

A practical rule: model as a star, measure, and add wide tables or aggregates as marts where a measured query
needs them. Physical design follows the queries, so it is decided after the model, not instead of it.

**Where it matters**
- Deciding between star schemas, wide tables and aggregates for a dashboard or a data product.

**Watch out for**
- Denormalizing everything up front for speed before measuring: you pay in flexibility for a gain you may not need.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** A BigQuery table is partitioned by `order_date`. A query filters `where extract(year from order_date) = 2025`. Why might it scan the whole table, and how do you fix it?

**A.** Pruning works on the partition column's values; wrapping the column in a function can prevent it. Filter on a range instead: `order_date >= '2025-01-01' and order_date < '2026-01-01'`.

**Q2.** Why does sorting a file by date make a date-range query cheaper even without partitioning?

**A.** Each row group's min and max dates become narrow, so the engine can skip every row group whose range does not overlap the filter. Unsorted, every row group spans all dates.

**Q3.** A team partitions a 2 GB table by `customer_id` (200,000 customers). What problem do they create?

**A.** Hundreds of thousands of tiny files: listing and opening them dominates query time, and compression and statistics suffer. Partition by date, or cluster by customer instead.

**Q4.** A daily aggregate stores `avg_order_value` per day. What goes wrong when a dashboard shows it by month?

**A.** Averaging daily averages weights every day equally regardless of order count. Store revenue and order count, sum both by month, then divide.

**Q5.** Why do two runs of the same query over a `double` amount column sometimes return different totals?

**A.** Floating-point addition rounds at each step, so the result depends on the order of additions, which varies with parallel execution. `decimal` sums are exact.

## Key takeaways

- Columnar engines read only the columns and blocks a query needs; physical design decides how many blocks that is.
- Partition by a column almost every query filters on, usually a date; avoid many tiny partitions.
- Filter on the partition column itself, or use hidden partitioning, so pruning happens.
- Sort or cluster so zone maps can skip blocks: the same query can read a tenth of the data.
- Aggregate tables store additive measures at a coarser, conformed grain; read them with `sum`, not `count`.
- Use integer surrogate keys and `decimal` for money; floating-point sums change with the order of addition.

**Next:** Continue with [Data modeling 9: testing, metrics and change](data-modeling-09-testing-metrics-and-change.ipynb).